# QCNN Benchmark - Ansatz & Encoding Matrix Study
**Fashion-MNIST 4-class Classification with Quantum Convolutional Neural Networks**

---

## Abstract

This notebook reports a systematic benchmark of **Quantum Convolutional Neural Networks
(QCNNs)** on a 4-class Fashion-MNIST task, sweeping across every combination of:

| Axis | Variants |
|------|---------|
| **Convolutional ansatz** | `hur6` (6 p) / `hur8` (10 p) / `hur9` (15 p) / `custom` (11 p) |
| **Data encoding** | `e3` amplitude / `e1` affine-angle + ancilla / `custom` fragment re-upload |
| **Random seeds** | 42 / 43 / 44 |

Up to **36 independent training runs** (4 x 3 x 3). The notebook is fully self-contained:
all circuit definitions are inlined, no external project modules are imported.

> **Dataset**: Fashion-MNIST / classes T-shirt/top (0), Trouser (1), Sneaker (7), Bag (8)
> / 16 x 16 pixels / **split**: 12 000 train / 2 000 val / 2 000 test (500/class each)

## Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Research Background](#2-research-background)
3. [Dataset Overview](#3-dataset-overview)
4. [Model Architecture](#4-model-architecture)
5. [Running the Experiments](#5-running-the-experiments) *(instructions - skip if results exist)*
6. [Loading Results](#6-loading-results)
7. [Comparative Analysis](#7-comparative-analysis)
   - 7.1a Seed Variability (Box Plot)
   - 7.1b Encoding Comparison (best ansatz per encoding)
   - 7.1c Ansatz Comparison (best encoding per ansatz)
   - 7.2 Ansatz x Encoding Heatmap
   - 7.3 Training Dynamics
   - 7.4 Confusion Matrices
   - 7.5 Per-class Accuracy
   - 7.6 Parameter Efficiency
8. [Key Findings](#8-key-findings)

## 1. Environment Setup

Install all required packages. Analysis (Sections 6-8) runs on CPU; GPU is only needed for training (Section 5).

In [ ]:
# Install all required packages (re-run if any import fails).
%pip install -q pennylane==0.45.0 pennylane-lightning==0.45.0 \
              torch torchvision \
              numpy pandas matplotlib seaborn scipy

In [ ]:
import os, sys, json, glob, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display
warnings.filterwarnings("ignore")

plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": "white",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 11})
sns.set_palette("Set2")
COLORS = sns.color_palette("Set2", 8)

# -- Paths (notebook assumed to be run from repository root) ------------------
ROOT       = Path.cwd()
RESULTS    = ROOT / "suitev2" / "results"
SUMMARY    = RESULTS / "summary.csv"

CLASS_NAMES  = ["T-shirt", "Trouser", "Sneaker", "Bag"]
ANSATZ_ORDER = ["hur6", "hur8", "hur9", "custom_ansatz"]
ENC_ORDER    = ["e3", "e1", "custom"]
ANSATZ_COLORS = {a: COLORS[i] for i, a in enumerate(ANSATZ_ORDER)}

print(f"Results : {RESULTS}")
print(f"summary : {'EXISTS' if SUMMARY.exists() else 'NOT FOUND - see Section 5'}")

## 2. Research Background

### 2.1 Quantum Convolutional Neural Networks

The QCNN architecture (Cong et al. 2019; Hur et al. 2022) mirrors classical CNN structure on
quantum circuits:

- **Convolutional layers** - parametric two-qubit unitary gates applied translationally over
  neighbouring qubit pairs (weight-shared, like a sliding filter window).
- **Pooling layers** - controlled rotations reducing active qubits by half (8 -> 4 -> 2).
- **Readout** - `probs(wires=[0, 4])` yields 4 class probabilities.

Full pipeline on 8 qubits:
```
Input image -> Encoding -> Conv1 (8 wires) -> Pool1 (8->4) -> Conv2 (4 wires) -> Pool2 (4->2) -> probs
```

### 2.2 Hur et al. (2022) Reference

The paper proposes `hur8` (10 parameters) and reports ~85 % accuracy on Fashion-MNIST
4-class at 16 x 16. This benchmark extends that baseline by varying both the ansatz and the
encoding strategy in a controlled matrix experiment.

### 2.3 Research Questions

> 1. Does a more expressive SU(4) ansatz (`hur9`, 15 params) outperform the simpler baseline?
> 2. Do learned re-uploading encodings (`e1`, `custom`) provide an advantage over simple
>    amplitude embedding (`e3`)?
> 3. What is the most parameter-efficient (ansatz, encoding) combination?

## 3. Dataset Overview

| Label | Class | Train | Val | Test |
|-------|-------|------:|----:|-----:|
| 0 | T-shirt/top | 3 000 | 500 | 500 |
| 1 | Trouser | 3 000 | 500 | 500 |
| 7 | Sneaker | 3 000 | 500 | 500 |
| 8 | Bag | 3 000 | 500 | 500 |

Images are down-sampled from 28 x 28 to **16 x 16** and normalised to [0, 1].
Random-chance accuracy for the balanced 4-class problem is **25 %**.

In [ ]:
import torch, torchvision
import torchvision.transforms as T

_ds = torchvision.datasets.FashionMNIST(
    root=str(ROOT / ".cache" / "fmnist"), train=True, download=True,
    transform=T.Compose([T.Resize(16), T.ToTensor()]))

TARGET = {0: "T-shirt", 1: "Trouser", 7: "Sneaker", 8: "Bag"}
samples = {k: [] for k in TARGET}
for img, lbl in _ds:
    if lbl in samples and len(samples[lbl]) < 3:
        samples[lbl].append(img.squeeze().numpy())
    if all(len(v) == 3 for v in samples.values()):
        break

fig, axes = plt.subplots(4, 3, figsize=(6, 8))
for row, lbl in enumerate(TARGET):
    for col in range(3):
        ax = axes[row, col]
        ax.imshow(samples[lbl][col], cmap="gray", interpolation="nearest")
        ax.axis("off")
        if col == 0:
            ax.set_ylabel(TARGET[lbl], fontsize=12, rotation=90, va="center")
fig.suptitle("Fashion-MNIST - 4 selected classes at 16x16", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

## 4. Model Architecture

### 4.1 Ansatz Circuits

| Ansatz | Type | Params | Gate structure |
|--------|------|:------:|----------------|
| `hur6` | SO(4) | 6 | RY-RY - CNOT - RY-RY - CNOT - RY-RY |
| `hur8` | Hur baseline | 10 | RX-RX - RZ-RZ - RX-RX - CNOT - RX-RX - RZ-RZ |
| `hur9` | SU(4)/KAK | 15 | U3-U3 - CNOT - RY-RZ - CNOT - RY - CNOT - U3-U3 |
| `custom` | Cartan | 11 | RZ-RY (x2) - IsingXX/YY/ZZ - RY-RZ (x2) |

Conv layer: same parameter vector shared across all qubit pairs (translational invariance).
Pooling: `hur_pool_pair` (CRZ + open-CRX, **2 params**) for hur6/8/9;
`transfer_pooling_pair` (CNOT-CRY-CRZ-RY, **3 params**) for custom.

### 4.2 Encoding Strategies

| Encoding | Qubits | Extra params | Description |
|----------|:------:|:------------:|-------------|
| `e3` | 8 | 0 | `AmplitudeEmbedding` - 256 normalised pixel amplitudes |
| `e1` | 9 | 16 | Affine angle encoding on 8 qubits + global ancilla (wire 8); re-uploads 4 global statistics (mean, variance, gradient energy, H/V asymmetry) via CNOT-RZ-CNOT fusion |
| `custom` | 8 | 96 | Pairwise fragment re-uploading: 4 image patches x 4 re-upload steps x 4 learned affine-mapped features |

### 4.3 Total Trainable Parameters

Parameters are weight-shared: each of the four layers (`conv1`, `pool1`, `conv2`, `pool2`)
uses one shared parameter vector. Total = 2 x `n_conv` + 2 x `n_pool` + `n_encoding`.

| Ansatz | n_conv | n_pool | +e3 (0) | +e1 (16) | +custom (96) |
|--------|:------:|:------:|:-------:|:--------:|:------------:|
| `hur6` | 6 | 2 | 16 | 32 | 112 |
| `hur8` | 10 | 2 | 24 | 40 | 120 |
| `hur9` | 15 | 2 | 34 | 50 | 130 |
| `custom` | 11 | 3 | 28 | 44 | 124 |

In [ ]:
import pennylane as qml

# -- Ansatz circuits (inlined from suitev2/ansatz.py) -------------------------

def hur6(theta, wires):
    """SO(4) circuit, 6 parameters."""
    a, b = wires
    qml.RY(theta[0], wires=a); qml.RY(theta[1], wires=b)
    qml.CNOT(wires=[a, b])
    qml.RY(theta[2], wires=a); qml.RY(theta[3], wires=b)
    qml.CNOT(wires=[a, b])
    qml.RY(theta[4], wires=a); qml.RY(theta[5], wires=b)

def hur8(theta, wires):
    """Hur 2022 baseline circuit, 10 parameters."""
    a, b = wires
    qml.RX(theta[0], wires=a); qml.RX(theta[1], wires=b)
    qml.RZ(theta[2], wires=a); qml.RZ(theta[3], wires=b)
    qml.RX(theta[4], wires=a); qml.RX(theta[5], wires=b)
    qml.CNOT(wires=[a, b])
    qml.RX(theta[6], wires=a); qml.RX(theta[7], wires=b)
    qml.RZ(theta[8], wires=a); qml.RZ(theta[9], wires=b)

def hur9(theta, wires):
    """SU(4) / KAK circuit, 15 parameters."""
    a, b = wires
    qml.U3(theta[0], theta[1], theta[2], wires=a)
    qml.U3(theta[3], theta[4], theta[5], wires=b)
    qml.CNOT(wires=[a, b])
    qml.RY(theta[6], wires=a); qml.RZ(theta[7], wires=b)
    qml.CNOT(wires=[b, a]); qml.RY(theta[8], wires=a)
    qml.CNOT(wires=[a, b])
    qml.U3(theta[9],  theta[10], theta[11], wires=a)
    qml.U3(theta[12], theta[13], theta[14], wires=b)

def custom_ansatz(theta, wires):
    """Cartan-inspired circuit, 11 parameters."""
    a, b = wires
    qml.RZ(theta[0], wires=a); qml.RY(theta[1], wires=a)
    qml.RZ(theta[2], wires=b); qml.RY(theta[3], wires=b)
    qml.IsingXX(theta[4], wires=[a, b])
    qml.IsingYY(theta[5], wires=[a, b])
    qml.IsingZZ(theta[6], wires=[a, b])
    qml.RY(theta[7], wires=a); qml.RZ(theta[8], wires=a)
    qml.RY(theta[9], wires=b); qml.RZ(theta[10], wires=b)

def conv_layer(conv_fn, theta, active_wires):
    """Translationally invariant conv: even + shifted pairs, shared params."""
    n = len(active_wires)
    even    = [(active_wires[i], active_wires[i+1]) for i in range(0, n-1, 2)]
    shifted = [(active_wires[i], active_wires[i+1]) for i in range(1, n-1, 2)]
    for pair in even + shifted:
        conv_fn(theta, pair)

print("Ansatz circuits defined.")

In [ ]:
# -- Visualise hur8 on 2 qubits -----------------------------------------------
dev = qml.device("default.qubit", wires=2)

@qml.qnode(dev)
def _draw_hur8(theta):
    hur8(theta, wires=[0, 1])
    return qml.state()

theta_zero = np.zeros(10)
print("hur8 convolutional ansatz (10 params, applied to each pair of neighbouring qubits):")
print(qml.draw(_draw_hur8)(theta_zero))

@qml.qnode(dev)
def _draw_hur6(theta):
    hur6(theta, wires=[0, 1])
    return qml.state()
print("\nhur6 ansatz (6 params, SO(4)):")
print(qml.draw(_draw_hur6)(np.zeros(6)))

## 5. Running the Experiments

> **Skip this section if `suitev2/results/summary.csv` already exists.**

Run the following commands **from the repository root in a terminal**:

```bash
# Full suite - all (ansatz, encoding, seed) combinations, 20 epochs each
python -m suitev2.run_suite

# Smoke test - fast pipeline check (2 epochs, 50 samples/class, ~5 min)
python -m suitev2.run_suite --epochs 2 --train-per-class 50

# Single combination (e.g., hur8 + amplitude encoding, all seeds)
python -m suitev2.run_suite --combo hur8_e3

# Extend an existing run by 10 more epochs
python -m suitev2.extend_run --run-dir suitev2/results/hur8_e3_seed42 --mode train --extra 10
```

The suite is **resumable**: a run whose `test.json` already exists is automatically skipped.
Results are written incrementally to `suitev2/results/summary.csv` (one row per run).

**Estimated wall-clock time** (GPU):
- `e3` encoding: ~3-5 min/run / `e1`/`custom`: ~8-15 min/run
- Full 27-run suite: **~3-6 hours** on a modern GPU.

## 6. Loading Results

The cells below load `suitev2/results/summary.csv` (one row per completed run) and
the per-run `metrics.csv` files (one row per training epoch). Numeric fields stored as
strings in the CSV are coerced to float. Extension rows (`chunk_id != 'base'`) are
filtered out so that only the original base training runs are included in the analysis.

In [ ]:
df_raw = None
df     = None   # base runs only

if not SUMMARY.exists():
    print("WARNING:  summary.csv not found - see Section 5.")
else:
    df_raw = pd.read_csv(SUMMARY)
    if df_raw.empty:
        print("WARNING:  summary.csv is empty.")
    else:
        df = df_raw[df_raw.get("chunk_id", "base") == "base"].copy() \
             if "chunk_id" in df_raw.columns else df_raw.copy()
        for c in ["test_acc","val_acc_best","val_acc_final","train_acc_final",
                  "test_loss","best_val_loss","n_params","best_epoch","n_epochs_trained"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
        df["combo"] = df["ansatz"] + "+" + df["encoding"]
        print(f"[OK]  {len(df)} base runs  |  "
              f"{df.groupby(['ansatz','encoding']).ngroups} combos  |  "
              f"seeds: {sorted(df['seed'].unique())}")
        display(df[["ansatz","encoding","seed","n_params","n_epochs_trained",
                     "test_acc","val_acc_best","best_epoch"]].sort_values(
                     ["ansatz","encoding","seed"]).to_string(index=False))

In [ ]:
# -- Early-stopping summary ----------------------------------------------------
if df is not None and not df.empty:
    es = df.groupby(["ansatz","encoding"]).agg(
        mean_epochs=("n_epochs_trained","mean"),
        min_epochs=("n_epochs_trained","min"),
        max_epochs=("n_epochs_trained","max"),
        mean_best_epoch=("best_epoch","mean"),
    ).reset_index()
    es["stopped_early"] = es["max_epochs"] < 20
    print("Early-stopping summary (max configured epochs = 20):")
    display(es.round(1).to_string(index=False))

In [ ]:
# -- Load per-run training curves ---------------------------------------------
curves = {}   # combo_key -> list of DataFrames
if RESULTS.exists():
    for p in sorted(RESULTS.glob("*/metrics.csv")):
        run = p.parent.name          # e.g. "hur8_e3_seed42"
        parts = run.rsplit("_seed", 1)
        if len(parts) == 2:
            key = parts[0]
            try:
                c = pd.read_csv(p)
                for col in ["train_loss","train_acc","val_loss","val_acc"]:
                    if col in c.columns:
                        c[col] = pd.to_numeric(c[col], errors="coerce")
                curves.setdefault(key, []).append(c)
            except Exception:
                pass
print(f"Training curves loaded: {sum(len(v) for v in curves.values())} runs, "
      f"{len(curves)} combos")

## 7. Comparative Analysis

### 7.1 Accuracy Analysis

Three complementary visualisations are produced in the cell below.

**7.1a - Seed Variability (Box Plot)**: one box per (ansatz, encoding) combination showing
the distribution across the 3 seeds. The box spans the interquartile range; the horizontal
line is the median; dots are individual seed results. This plot reveals how stable each
configuration is across different data splits.

**7.1b - Encoding Comparison**: one bar per encoding type, shown at its *best-performing
ansatz*. Encoding and ansatz are independent design axes; averaging across all ansatz for
a given encoding would conflate the two effects. Showing the best ansatz per encoding
gives the upper bound for each encoding strategy and allows a fair, isolated comparison.

**7.1c - Ansatz Comparison**: one bar per ansatz circuit, shown at its *best-performing
encoding*. Symmetric to 7.1b: the best encoding per ansatz isolates the ansatz effect
without penalising an ansatz for a suboptimal encoding choice.

All bars show mean +/- std over the 3 seeds.

In [ ]:
def _no_data(s): print(f"WARNING:  No data for {s} - run Section 5 then reload Section 6.")

if df is None:
    _no_data("7.1")
else:
    grp = df.groupby(["ansatz","encoding"])["test_acc"].agg(["mean","std"]).reset_index()
    grp["std"] = grp["std"].fillna(0)
    grp = grp.sort_values("mean", ascending=False).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(max(10, len(grp)), 5))
    ax.bar(range(len(grp)), grp["mean"], yerr=grp["std"], capsize=5,
           color=[ANSATZ_COLORS.get(a, COLORS[0]) for a in grp["ansatz"]],
           edgecolor="white", alpha=0.85)
    ax.axhline(0.25, ls="--", color="gray", lw=1.2, label="Random chance")
    ax.set_xticks(range(len(grp)))
    ax.set_xticklabels(grp["ansatz"] + "\n" + grp["encoding"], fontsize=9)
    ax.set_ylabel("Mean Test Accuracy"); ax.set_ylim(0, 1.08)
    ax.set_title("Mean Test Accuracy per (Ansatz, Encoding) - sorted descending", fontsize=13)
    for i, r in grp.iterrows():
        ax.text(i, r["mean"] + r["std"] + 0.01, f'{r["mean"]:.3f}',
                ha="center", fontsize=8)
    handles = [mpatches.Patch(color=ANSATZ_COLORS[a], label=a)
               for a in ANSATZ_ORDER if a in grp["ansatz"].values]
    handles.append(plt.Line2D([0],[0], ls="--", color="gray", label="Random chance"))
    ax.legend(handles=handles, fontsize=9, loc="lower right")
    plt.tight_layout(); plt.show()
    pivot = grp.pivot_table(index="ansatz", columns="encoding", values="mean", aggfunc="first")
    print("\nMean test accuracy pivot table:")
    display(pivot.round(4))

### 7.2 Ansatz x Encoding Heatmap

Darker = higher mean test accuracy. Cells without results appear masked.

In [ ]:
if df is None:
    _no_data("7.2")
else:
    grp2 = df.groupby(["ansatz","encoding"])["test_acc"].mean().reset_index()
    pivot2 = grp2.pivot(index="ansatz", columns="encoding", values="test_acc")
    pa = [a for a in ANSATZ_ORDER if a in pivot2.index]
    pe = [e for e in ENC_ORDER if e in pivot2.columns]
    pivot2 = pivot2.reindex(index=pa, columns=pe)

    cnt = df.groupby(["ansatz","encoding"])["seed"].count().reset_index()
    cnt = cnt.pivot(index="ansatz", columns="encoding", values="seed").reindex(index=pa, columns=pe)

    annot = pivot2.copy().astype(object)
    for i in pa:
        for j in pe:
            v = pivot2.loc[i, j]; n = int(cnt.loc[i, j]) if not pd.isna(cnt.loc[i, j]) else 0
            annot.loc[i, j] = f"{v:.3f}\n(n={n})" if not pd.isna(v) else "-"

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.heatmap(pivot2, annot=annot, fmt="", cmap="YlGnBu", vmin=0.25, vmax=1.0,
                linewidths=0.5, ax=ax, mask=pivot2.isna(),
                cbar_kws={"label": "Mean Test Accuracy"}, annot_kws={"size": 10})
    ax.set_title("Mean Test Accuracy: Ansatz x Encoding", fontsize=13, pad=10)
    ax.set_xlabel("Encoding"); ax.set_ylabel("Ansatz")
    plt.tight_layout(); plt.show()

### 7.3 Training Dynamics

For each combination: **train accuracy** (solid) and **validation accuracy** (dashed), mean +/- std over seeds.
A large train-val gap indicates overfitting; slow convergence suggests underfitting or a difficult landscape.

In [ ]:
if not curves:
    _no_data("7.3 - no metrics.csv files found")
else:
    keys = sorted(curves.keys())
    ncols = min(3, len(keys))
    nrows = (len(keys) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5*ncols, 3.5*nrows), squeeze=False)
    axf = axes.flatten()

    for idx, key in enumerate(keys):
        ax = axf[idx]
        seed_dfs = curves[key]
        max_ep = max(c["epoch"].max() for c in seed_dfs)
        train_mat = np.full((len(seed_dfs), int(max_ep)), np.nan)
        val_mat   = np.full((len(seed_dfs), int(max_ep)), np.nan)
        for si, c in enumerate(seed_dfs):
            ep = c["epoch"].astype(int).values - 1
            for e, ta, va in zip(ep, c["train_acc"].values, c["val_acc"].values):
                if 0 <= e < int(max_ep):
                    train_mat[si, e] = ta
                    val_mat[si, e]   = va

        x = np.arange(1, int(max_ep)+1)
        ans_part = key.split("_")[0]
        color = ANSATZ_COLORS.get(ans_part, COLORS[0])

        m_tr, s_tr = np.nanmean(train_mat, 0), np.nanstd(train_mat, 0)
        m_va, s_va = np.nanmean(val_mat,   0), np.nanstd(val_mat,   0)

        ax.plot(x, m_tr, color=color, lw=2, label="train acc")
        ax.fill_between(x, m_tr-s_tr, m_tr+s_tr, alpha=0.15, color=color)
        ax.plot(x, m_va, color=color, lw=2, ls="--", label="val acc")
        ax.fill_between(x, m_va-s_va, m_va+s_va, alpha=0.10, color=color)
        ax.axhline(0.25, ls=":", color="gray", lw=0.8, alpha=0.5)
        ax.set_title(key.replace("_", "+", 1), fontsize=10)
        ax.set_xlabel("Epoch", fontsize=9); ax.set_ylabel("Accuracy", fontsize=9)
        ax.set_ylim(0, 1.05); ax.tick_params(labelsize=8)
        if idx == 0:
            ax.legend(fontsize=7, loc="lower right")

    for ax in axf[len(keys):]: ax.set_visible(False)
    fig.suptitle("Train (solid) vs Val (dashed) Accuracy - mean+/-std over seeds", fontsize=13, y=1.01)
    plt.tight_layout(); plt.show()

### 7.4 Confusion Matrices

One matrix per (ansatz, encoding) combination, **summed over seeds**.
Cells show absolute counts; colour shows row-normalised fraction.

In [ ]:
def _extract_cm(row, n=4):
    cm = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(n):
            col = f"cm_{i}{j}"
            if col in row.index and not pd.isna(row[col]):
                cm[i, j] = int(row[col])
    return cm

def _plot_cm(ax, cm, title):
    norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    sns.heatmap(norm, annot=cm, fmt="d", cmap="Blues", vmin=0, vmax=1,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, cbar=False, annot_kws={"size": 9})
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("Predicted", fontsize=8); ax.set_ylabel("True", fontsize=8)
    ax.tick_params(labelsize=8)

if df is None:
    _no_data("7.4")
else:
    combos = sorted(df.groupby(["ansatz","encoding"]).groups.keys())
    ncols = min(3, len(combos))
    nrows = (len(combos) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows), squeeze=False)
    axf = axes.flatten()
    for idx, (ans, enc) in enumerate(combos):
        sub = df[(df["ansatz"]==ans) & (df["encoding"]==enc)]
        cm_tot = sum(_extract_cm(r) for _, r in sub.iterrows())
        _plot_cm(axf[idx], cm_tot, f"{ans}+{enc}  (n={len(sub)})")
    for ax in axf[len(combos):]: ax.set_visible(False)
    fig.suptitle("Confusion Matrices - summed over seeds", fontsize=13, y=1.01)
    plt.tight_layout(); plt.show()

### 7.5 Per-class Accuracy

Per-class accuracy from confusion-matrix diagonal. Reveals which classes are systematically harder.

In [ ]:
if df is None:
    _no_data("7.5")
else:
    rows = []
    for ans, enc in sorted(df.groupby(["ansatz","encoding"]).groups.keys()):
        sub = df[(df["ansatz"]==ans) & (df["encoding"]==enc)]
        cm = sum(_extract_cm(r) for _, r in sub.iterrows())
        acc = cm.diagonal() / cm.sum(axis=1).clip(min=1)
        rows.append({"combo": f"{ans}+{enc}", **{CLASS_NAMES[k]: acc[k] for k in range(4)}})
    df_pc = pd.DataFrame(rows)

    x = np.arange(len(df_pc)); w = 0.2
    cls_colors = ["#4C72B0","#DD8452","#55A868","#C44E52"]
    fig, ax = plt.subplots(figsize=(max(10, len(df_pc)*1.2), 5))
    for k, cn in enumerate(CLASS_NAMES):
        ax.bar(x + k*w, df_pc[cn], w, label=cn, color=cls_colors[k], alpha=0.85)
    ax.set_xticks(x + 1.5*w)
    ax.set_xticklabels(df_pc["combo"], rotation=25, ha="right", fontsize=9)
    ax.set_ylabel("Per-class Accuracy"); ax.set_ylim(0, 1.1)
    ax.set_title("Per-class Accuracy by Configuration", fontsize=13)
    ax.legend(title="Class", fontsize=9)
    plt.tight_layout(); plt.show()
    print("\nPer-class accuracy table:")
    display(df_pc.round(3).to_string(index=False))

### 7.6 Parameter Efficiency

Scatter of mean test accuracy vs. total trainable parameters. Top-left = best efficiency.

In [ ]:
if df is None:
    _no_data("7.6")
else:
    grp_pe = df.groupby(["ansatz","encoding"]).agg(
        mean_acc=("test_acc","mean"), std_acc=("test_acc","std"),
        n_params=("n_params","first")).reset_index()
    enc_markers = {"e3":"o","e1":"s","custom":"^"}

    fig, ax = plt.subplots(figsize=(8, 5))
    for _, r in grp_pe.iterrows():
        m = enc_markers.get(r["encoding"], "D")
        c = ANSATZ_COLORS.get(r["ansatz"], COLORS[0])
        ax.scatter(r["n_params"], r["mean_acc"], s=120, marker=m, color=c,
                   alpha=0.85, zorder=3, edgecolors="white", lw=0.8)
        ax.errorbar(r["n_params"], r["mean_acc"],
                    yerr=r["std_acc"] if not pd.isna(r["std_acc"]) else 0,
                    fmt="none", color=c, capsize=4, lw=1.5, alpha=0.6)
        ax.annotate(f'{r["ansatz"]}\n{r["encoding"]}',
                    (r["n_params"], r["mean_acc"]),
                    textcoords="offset points", xytext=(6,4), fontsize=7)
    ax.axhline(0.25, ls="--", color="gray", lw=1, label="Random chance")
    ax.set_xlabel("Total Trainable Parameters"); ax.set_ylabel("Mean Test Accuracy")
    ax.set_ylim(0, 1.05)
    ax.set_title("Parameter Efficiency: Accuracy vs. Parameters", fontsize=13)
    ans_h = [mpatches.Patch(color=ANSATZ_COLORS[a], label=a)
             for a in ANSATZ_ORDER if a in grp_pe["ansatz"].values]
    enc_h = [plt.scatter([],[], marker=enc_markers[e], color="gray", s=80, label=f"enc={e}")
             for e in ENC_ORDER if e in grp_pe["encoding"].values]
    ax.legend(handles=ans_h+enc_h, fontsize=9, loc="lower right", ncol=2)
    plt.tight_layout(); plt.show()

## 8. Key Findings

This section synthesises the evidence from Sections 7.1-7.6. Specific numerical values
should be read from the plots; the framework below applies to any outcome of the
(ansatz, encoding) matrix.

### 8.1 Effect of Ansatz Expressivity

The four ansatz circuits cover a range from 6 parameters (hur6, SO(4) subgroup) to
15 parameters (hur9, full SU(4)). Three possible outcomes are interpretable:

- **Monotone improvement with parameters** (hur9 > hur8 > hur6): expressivity matters
  for this task and the convolution layer is the performance bottleneck. The richer
  gate set of SU(4) provides a measurable advantage within 20 training epochs.
- **Plateau at hur8**: the Hur et al. (2022) baseline already captures the relevant
  unitary structure; additional parameters do not contribute. This would suggest that
  the task complexity, rather than the circuit expressivity, is the limiting factor.
- **Non-monotone ranking**: the optimal ansatz depends on the encoding, indicating an
  interaction effect. In this case, the (ansatz, encoding) pair should be treated as
  a single design choice, not two independent axes. Inspect the heatmap (Section 7.2).

### 8.2 Effect of Encoding Strategy

The three encodings differ structurally:

- **e3 (AmplitudeEmbedding, 0 extra params)**: discards all spatial structure by mapping
  256 pixel values to 256 complex amplitudes. Provides a clean no-overhead baseline.
- **e1 (affine re-upload, +16 params)**: injects four global image statistics (mean,
  pixel variance, gradient energy, horizontal/vertical asymmetry) via a re-uploading
  scheme on the ancilla qubit. Useful if those statistics are discriminative for the
  T-shirt / Trouser / Sneaker / Bag problem.
- **custom (fragment re-upload, +96 params)**: applies learned pairwise fragment
  re-uploading over four image patches. The large parameter overhead raises the total
  count substantially (see Section 7.6); the gain must be weighed against the cost.

If e3 matches or exceeds e1 and custom in test accuracy, the raw pixel amplitudes
contain sufficient information and the re-uploading overhead is not justified.

### 8.3 Convergence and Stability

Early stopping (patience 10 epochs) under a cosine warm-restart schedule means that
configurations differ in the number of epochs actually trained. The early-stopping
summary (Section 6) reports this per combination. A configuration that consistently
stops early (e.g., at epoch 8-10) has found a stable minimum; one that always runs
to epoch 20 may still be improving and could benefit from training extension via
`python -m suitev2.extend_run`.

Seed-to-seed variance in test accuracy (shown as error bars in Section 7.1) quantifies
sensitivity to data-split randomness. High variance indicates that 3 seeds may be
insufficient to draw strong conclusions for that configuration.

### 8.4 Per-class Analysis

Fashion-MNIST class confusability is not uniform. Trouser (class 1) is typically the
easiest to identify due to its distinctive silhouette. T-shirt/top (class 0) and
Sneaker (class 7) are the most commonly confused pair, as both can exhibit similar
colour distributions at 16x16 resolution. Configurations that improve the Sneaker /
T-shirt boundary are of particular interest. Per-class accuracy is reported in
Section 7.5; the full confusion matrices are in Section 7.4.

### References
- Cong, I., Choi, S., & Lukin, M. D. (2019). *Quantum convolutional neural networks.*
  Nature Physics, 15, 1273-1278.
- Hur, T., Kim, L., & Park, D. K. (2022). *Quantum convolutional neural network for
  classical data classification.* Quantum Machine Intelligence, 4(1), 3.
- Loshchilov, I. & Hutter, F. (2017). *SGDR: Stochastic gradient descent with warm
  restarts.* International Conference on Learning Representations (ICLR).
- Szegedy, C. et al. (2016). *Rethinking the Inception architecture for computer vision.*
  IEEE Conference on Computer Vision and Pattern Recognition (CVPR).